# Loading Hospital Data from CSV Files


In [0]:
%python

# Read the Hospital General Information CSV
hospital_info = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Hospital_General_Information.csv",
    header=True,
    inferSchema=True
)

# Create a temporary view for SQL queries
hospital_info.createOrReplaceTempView("hospital_info")

print(f"Loaded {hospital_info.count()} rows")
display(hospital_info.limit(5))

In [0]:
%python
# Read the Complications and Deaths CSV
complications = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Complications_and_Deaths-Hospital.csv",
    header=True,
    inferSchema=True
)

complications.createOrReplaceTempView("complications")
print(f"Loaded {complications.count()} rows")

In [0]:
%python
# Read the HCAHPS Patient Survey CSV
patient_survey = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/HCAHPS-Hospital_Patient_Survey.csv",
    header=True,
    inferSchema=True
)

patient_survey.createOrReplaceTempView("patient_survey")
print(f"Loaded {patient_survey.count()} rows")

In [0]:
%python
# Read the Healthcare Associated Infections CSV
infections = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Healthcare_Associated_Infections-Hospital.csv",
    header=True,
    inferSchema=True
)

infections.createOrReplaceTempView("infections")
print(f"Loaded {infections.count()} rows")

In [0]:
%python
# Read the Medicare Hospital Spending CSV
medicare_spending = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Medicare_Hospital_Spending_by_Claim.csv",
    header=True,
    inferSchema=True
)

medicare_spending.createOrReplaceTempView("medicare_spending")
print(f"Loaded {medicare_spending.count()} rows")

In [0]:
%python
# Read the Timely and Effective Care CSV
timely_care = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Timely_and_Effective_Care-Hospital.csv",
    header=True,
    inferSchema=True
)

timely_care.createOrReplaceTempView("timely_care")
print(f"Loaded {timely_care.count()} rows")

In [0]:
%python
# Read the Unplanned Hospital Visits CSV
unplanned_visits = spark.read.csv(
    "/Volumes/workspace/default/hospital_assessment/Unplanned_Hospital_Visits-Hospital.csv",
    header=True,
    inferSchema=True
)

unplanned_visits.createOrReplaceTempView("unplanned_visits")
print(f"Loaded {unplanned_visits.count()} rows")

## Available SQL Tables

* `hospital_info` - Hospital General Information
* `complications` - Complications and Deaths
* `patient_survey` - HCAHPS Patient Survey results
* `infections` - Healthcare Associated Infections
* `medicare_spending` - Medicare Hospital Spending by Claim
* `timely_care` - Timely and Effective Care measures
* `unplanned_visits` - Unplanned Hospital Visits
* `py_clean` - Cleaned hospital data with outcomes, costs, and value quadrants (2,445 hospitals)

# Cleaning and Manipulating the Data

## Layer 0: Raw normalization of the data

In [0]:
%sql
-- Layer 0: Raw normalization of the data

CREATE OR REPLACE TEMP VIEW raw_complications AS
SELECT `Facility ID` AS Facility_ID
, `Measure ID` AS Measure_ID
, Score AS Score
FROM complications
;

CREATE OR REPLACE TEMP VIEW raw_readmissions AS
SELECT `Facility ID` AS Facility_ID
, `Measure ID` AS Measure_ID
, Score AS Score
FROM unplanned_visits
;

CREATE OR REPLACE TEMP VIEW raw_infections AS
SELECT `Facility ID` AS Facility_ID
, `Measure ID` AS Measure_ID
, Score AS Score
FROM infections
;

CREATE OR REPLACE TEMP VIEW raw_te AS
SELECT `Facility ID` AS Facility_ID
, `Measure ID` AS Measure_ID
, Score AS Score
, Sample AS Sample
FROM timely_care
;

CREATE OR REPLACE TEMP VIEW raw_cost AS
SELECT `Facility ID` AS Facility_ID
, Period
, `Avg Spndg Per EP Hospital` AS Avg_Spnd_Per_EP
FROM medicare_spending
;

CREATE OR REPLACE TEMP VIEW raw_survey AS
SELECT `Facility ID` AS Facility_ID
, `HCAHPS Measure ID` AS Measure_ID
, `Patient Survey Star Rating` AS Score
FROM patient_survey
;

CREATE OR REPLACE TEMP VIEW raw_hospital AS
SELECT `Facility ID` AS Facility_ID
, `Facility Name` AS Facility_Name
, `City/Town` AS City_Town
, State
, `ZIP Code` AS ZIP_Code
, `Hospital Type` AS Hospital_Type
, `Hospital Ownership` AS Hospital_Ownership
, `Emergency Services` AS Emergency_Services
, `Hospital overall rating` AS Hospital_Overall_Rating
, `Count of Facility MORT Measures` AS Cnt_Mort
, `Count of Facility Safety Measures` AS Cnt_Safety
, `Count of Facility READM Measures` AS Cnt_Readm
, `Count of Facility Pt Exp Measures` AS Cnt_Ptexp
, `Count of Facility TE Measures` AS Cnt_Te
FROM hospital_info
;

## Layer 1: Calculating overall z-score

In [0]:
%sql
-- Layer 1: Putting measures together in one long table, z-scoring by domain, and calculating overall z-score
-- Note: CMS uses exactly the same measures to calculate outcomes scores for its star ratings.

-- one long table
CREATE OR REPLACE TEMP VIEW measures_long AS
SELECT Facility_ID
, Measure_ID
, CASE WHEN Measure_ID IN ('MORT_30_AMI','MORT_30_CABG', 
                    'MORT_30_COPD','MORT_30_HF',
                    'MORT_30_PN','MORT_30_STK','PSI_04')
        THEN 'mort'
        WHEN Measure_ID IN ('EDAC_30_AMI','EDAC_30_HF',
                     'EDAC_30_PN','READM_30_CABG',
                     'READM_30_COPD','READM_30_HIP_KNEE',
                     'Hybrid_HWR','OP_32','OP_35_ADM',
                     'OP_35_ED','OP_36')
        THEN 'readm'
        WHEN Measure_ID IN ('HAI_1_SIR','HAI_2_SIR',
                      'HAI_3_SIR','HAI_4_SIR',
                      'HAI_5_SIR','HAI_6_SIR', 
                      'COMP_HIP_KNEE', 'PSI_90')
        THEN 'safety'
    END AS Domain
, try_cast(Score AS double) AS Score
FROM  (
    SELECT Facility_ID
    , Measure_ID
    , Score
    FROM raw_complications
    UNION ALL
    SELECT Facility_ID
    , Measure_ID
    , Score
    FROM raw_readmissions
    UNION ALL
    SELECT Facility_ID
    , Measure_ID
    , Score
    FROM raw_infections
) AS U
WHERE Measure_ID IN ('MORT_30_AMI','MORT_30_CABG', 
                    'MORT_30_COPD','MORT_30_HF',
                    'MORT_30_PN','MORT_30_STK','PSI_04',
                    'EDAC_30_AMI','EDAC_30_HF',
                    'EDAC_30_PN','READM_30_CABG',
                    'READM_30_COPD','READM_30_HIP_KNEE',
                    'Hybrid_HWR','OP_32','OP_35_ADM',
                    'OP_35_ED','OP_36',
                    'HAI_1_SIR','HAI_2_SIR',
                    'HAI_3_SIR','HAI_4_SIR',
                    'HAI_5_SIR','HAI_6_SIR', 
                    'COMP_HIP_KNEE', 'PSI_90')
;

-- domain z-scores
CREATE OR REPLACE TEMP VIEW measure_z AS
SELECT Facility_ID
, Domain
, Measure_ID
, -1.0 * (Score - avg(Score) OVER (PARTITION BY Measure_ID)) / stddev_samp(Score) OVER (PARTITION BY Measure_ID) AS Z
FROM measures_long
;

-- average domain scores then overall z-score
CREATE OR REPLACE TEMP VIEW domain_master_z AS
WITH dmean AS (
    SELECT Facility_ID
    , Domain
    , avg(Z) AS Z_Mean
    FROM measure_z
    GROUP BY Facility_ID, Domain
    )
SELECT Facility_ID
, Domain
, (Z_Mean - avg(Z_Mean) OVER (PARTITION BY Domain)) / 
    stddev_samp(Z_Mean) OVER (PARTITION BY Domain) AS Master_Z
FROM dmean
;


## Layer 2: Calculating thresholds

In [0]:
%sql
-- Layer 2: Using the hospital data to calculate thresholds
-- Note: For the CMS star rating methodology, a hospital is included if it has 3 categories with at least 3 ratings per category, provided either Complications or Infections (or both) are represented. For this analysis, a hospital is included if it has at least 2 of the 3 categories (Complications, Infections, and Readmissions) with at least 3 ratings in the category.

CREATE OR REPLACE TEMP VIEW hospital_clean AS
WITH flagged AS (
    SELECT Facility_ID
    , Facility_Name
    , City_Town
    , State
    , lpad(CAST (ZIP_Code AS string),5,'0') AS ZIP_Code
    , try_cast(Hospital_Overall_Rating AS DOUBLE) AS Hospital_Overall_Rating
    , Hospital_Type
    , Hospital_Ownership
    , Emergency_Services
    , CASE WHEN try_cast(Cnt_Mort AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Mort_Flag
    , CASE WHEN try_cast(Cnt_Readm AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Readm_Flag
    , CASE WHEN try_cast(Cnt_Safety AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Safety_Flag
    , CASE WHEN try_cast(Cnt_Ptexp AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Ptexp_Flag
    , CASE WHEN try_cast(Cnt_Te AS DOUBLE) >= 3
            THEN 1
            ELSE 0
            END AS Te_Flag
    FROM raw_hospital
    )
SELECT *
, (Mort_Flag + Readm_Flag + Safety_Flag) AS Include_Score
, CASE (Ptexp_Flag + Te_Flag + Mort_Flag + Readm_Flag + Safety_Flag)
    WHEN 3
    THEN 'Small'
    WHEN 4
    THEN 'Medium'
    WHEN 5 THEN 'Large'
    END AS Size_Indicator
FROM flagged
WHERE (Mort_Flag + Readm_Flag + Safety_Flag) >= 2
QUALIFY ROW_NUMBER() OVER (PARTITION BY Facility_ID ORDER BY Facility_Name) = 1
;

## Layer 3: Creating the Outcomes score

In [0]:
%sql
-- Layer 3: Creating the Outcomes score by removing facilities with insufficient numbers of scores and averaging the remaining domain scores (Complications, Readmissions, and Infections) per facility

CREATE OR REPLACE TEMP VIEW outcomes AS
SELECT d.Facility_ID
, AVG(CASE d.Domain WHEN 'mort'
                    THEN CASE WHEN h.Mort_Flag = 1 
                            THEN d.Master_Z
                            END
                    WHEN 'readm'
                    THEN CASE WHEN h.Readm_Flag = 1
                            THEN d.Master_Z
                            END
                    WHEN 'safety'
                    THEN CASE WHEN h.Safety_Flag = 1
                            THEN d.Master_Z
                            END
                    END) AS Outcomes
FROM domain_master_z d
JOIN hospital_clean h
    ON d.Facility_ID = h.Facility_ID
GROUP BY d.Facility_ID
;

## Layer 4: Preparing Cost table

In [0]:
%sql
-- Layer 4: Preparing Cost table for joining by padding Facility_ID and creating Per_Episode_Cost column

CREATE OR REPLACE TEMP VIEW cost AS
SELECT lpad(cast(Facility_ID AS STRING), 6, '0') AS Facility_ID
, SUM(CASE WHEN Period = 'Complete Episode'
        THEN try_cast(Avg_Spnd_Per_Ep AS DOUBLE)
        ELSE 0
        END) AS Per_Episode_Cost
FROM raw_cost
GROUP BY lpad(cast(Facility_ID AS STRING), 6, '0')
;

## Layer 5: Creating Census Division and Value Quadrant and saving off table

In [0]:
%sql
-- Layer 5: Calculating Census Division from State and calculating Value Quadrant using median Outcomes and median Per_Episode_Cost
-- Note: facilities without cost information will be dropped because both Outcomes and Per_Episode_Cost are required for the Value Quadrant creation.

-- Creating Census Division 
CREATE OR REPLACE TEMP VIEW census_div (State, Census_Division) AS VALUES
('CT','New England'), ('ME','New England'), ('MA','New England'), ('NH','New England'), ('RI','New England'), ('VT','New England'),('NJ','Middle Atlantic'),('NY','Middle Atlantic'), ('PA','Middle Atlantic'),('IL','East North Central'),('IN','East North Central'),('MI','East North Central') ,('OH','East North Central'), ('WI','East North Central'), ('IA','West North Central'), ('KS','West North Central'), ('MN','West North Central'), ('MO','West North Central'), ('ND','West North Central'), ('SD','West North Central'), ('NE','West North Central'), ('DE','South Atlantic'),('DC','South Atlantic' ),('FL','South Atlantic'), ('GA','South Atlantic'),('MD','South Atlantic'),('NC','South Atlantic'),('SC','South Atlantic'),('VA', 'South Atlantic'),('WV','South Atlantic'), ('AL','East South Central'), ('KY','East South Central'), ('MS','East South Central'), ('TN','East South Central'), ('AR','West South Central'), ('LA','West South Central'), ('OK','West South Central'), ('TX','West South Central'), ('AZ','Mountain'), ('CO','Mountain'), ('ID','Mountain'), ('MT','Mountain'), ('NV','Mountain'),('NM','Mountain'), ('UT','Mountain'), ('WY','Mountain'), ('CA','Pacific'), ('HI','Pacific'),  ('OR','Pacific'), ('WA','Pacific'), ('AK', 'Pacific')
;

-- Joining Census Divisions and creating Value_Quadrant
CREATE OR REPLACE TEMP VIEW hospital_value AS
WITH base AS (
    SELECT h.Facility_ID
    , h.Facility_Name
    , h.City_Town
    , h.State
    , cd.Census_Division
    , h.ZIP_Code
    , h.Hospital_Overall_Rating
    , h.Hospital_Type
    , h.Hospital_Ownership
    , h.Emergency_Services
    , h.Size_Indicator
    , o.Outcomes
    , c.Per_Episode_Cost
    FROM hospital_clean h
    JOIN outcomes o
        ON h.Facility_ID = o.Facility_ID
    JOIN cost c
        ON h.Facility_ID = c.Facility_ID
    LEFT JOIN census_div cd
        ON h.State = cd.State
    )
, med AS (
    SELECT median(Outcomes) AS Outcome_Median
    , median(Per_Episode_Cost) AS Cost_Median
    FROM base
    )
SELECT b.*
, concat(CASE WHEN b.Outcomes > m.Outcome_Median
            THEN 'high outcome'
            ELSE 'low outcome'
            END
        , ' , ', 
        CASE WHEN b.Per_Episode_Cost < m.Cost_Median
            THEN 'low cost'
            ELSE 'high cost'
            END
        ) AS Value_Quadrant
FROM base b
CROSS JOIN med m
WHERE Facility_ID <> 10099. -- this is to match the dataset to the python one
;

-- Saving clean data table
CREATE OR REPLACE TABLE hospital_value_clean AS
SELECT *
FROM hospital_value
;








## Layer 6: Adding survey and timely & effective care measures

In [0]:
-- Layer 6: Survey and Timely & Effective care measures added to the clean data table

-- Event percentages of timely & effective care events
-- ED Wait Time and Left ED Not Seen are made negative so that higher will be better for all scores in the dataset
CREATE OR REPLACE TEMP VIEW te_wide AS
WITH te AS (
    SELECT Facility_ID
    , Measure_ID
    , CASE WHEN Measure_ID IN ('OP_22', 'OP_23', 'SEP_1')
        THEN try_cast(Score AS DOUBLE) / try_cast(Sample AS DOUBLE)
        ELSE try_cast (Score AS DOUBLE)
        END AS New_Score
    FROM raw_te
    WHERE Measure_ID IN ('OP_22', 'OP_23', 'SEP_1', 'OP_18b')
    )
SELECT Facility_ID
, MAX(CASE WHEN Measure_ID = 'SEP_1'
    THEN New_Score
    END) AS `Sepsis_Appropriate_Care`
, MAX(CASE WHEN Measure_ID = 'OP_23'
    THEN New_Score
    END) AS `Head_CT_Results_Quick`
, -1.0 * MAX(CASE WHEN Measure_ID = 'OP_18b' 
    THEN New_Score
    END) AS `ED_Wait_Time`
, -1.0 * MAX(CASE WHEN Measure_ID = 'OP_22'
    THEN New_Score
    END) AS `Left_ED_Not_Seen`
FROM te
GROUP BY Facility_ID
;

-- Survey data
CREATE OR REPLACE TEMP VIEW survey_wide AS
WITH survey AS (
    SELECT Facility_ID
    , Measure_ID
    , try_cast (Score AS DOUBLE) AS New_Score
    FROM raw_survey
    WHERE Measure_ID IN ('H_CLEAN_STAR_RATING',
                        'H_COMP_1_STAR_RATING',
                        'H_COMP_2_STAR_RATING', 
                        'H_COMP_5_STAR_RATING', 
                        'H_COMP_6_STAR_RATING', 
                        'H_QUIET_STAR_RATING', 
                        'H_RECMND_STAR_RATING')
    )
SELECT Facility_ID
, MAX(CASE WHEN Measure_ID = 'H_CLEAN_STAR_RATING'
    THEN New_Score
    END) AS `Cleanliness`
, MAX(CASE WHEN Measure_ID = 'H_COMP_1_STAR_RATING'
    THEN New_Score
    END) AS `Nurse_Communication`
, MAX(CASE WHEN Measure_ID = 'H_COMP_2_STAR_RATING'
    THEN New_Score
    END) AS `Doctor_Communication`
, MAX(CASE WHEN Measure_ID = 'H_COMP_5_STAR_RATING'
    THEN New_Score
    END) AS `Communication_About_Medicines`
, MAX(CASE WHEN Measure_ID = 'H_COMP_6_STAR_RATING'
    THEN New_Score
    END) AS `Discharge_Information`
, MAX(CASE WHEN Measure_ID = 'H_QUIET_STAR_RATING'
    THEN New_Score
    END) AS `Quietness`
, MAX(CASE WHEN Measure_ID = 'H_RECMND_STAR_RATING'
    THEN New_Score
    END) AS `Would_Recommend`
FROM survey
GROUP BY Facility_ID
;

-- Joining tables
CREATE OR REPLACE TEMP VIEW hospital_analysis AS
SELECT h.*
, t.`Sepsis_Appropriate_Care`
, t.`Head_CT_Results_Quick`
, t.`ED_Wait_Time`
, t.`Left_ED_Not_Seen`
, s.`Cleanliness`
, s.`Nurse_Communication`
, s.`Doctor_Communication`
, s.`Communication_About_Medicines`
, s.`Discharge_Information`
, s.`Quietness`
, s.`Would_Recommend`
FROM hospital_value h
LEFT JOIN te_wide t
    ON h.Facility_ID = t.Facility_ID
LEFT JOIN survey_wide s
    ON h.Facility_ID = s.Facility_ID
;

-- Saving clean data analysis table
CREATE OR REPLACE TABLE hospital_analysis_clean AS
SELECT *
FROM hospital_analysis
;

## Layer 7: Analysis aggregations

In [0]:
-- Analysis aggregations: categorical counts, continuous summary stats, net tilt by census division and ownership

-- Categorical counts
SELECT Census_Division
, COUNT(*) AS n
FROM hospital_analysis
GROUP BY Census_Division
ORDER BY n DESC
;



In [0]:
SELECT Hospital_Type
, COUNT(*) AS n
FROM hospital_analysis
GROUP BY Hospital_Type
ORDER BY n DESC
;



In [0]:
SELECT Hospital_Ownership
, COUNT(*) AS n
FROM hospital_analysis
GROUP BY Hospital_Ownership
ORDER BY n DESC
;



In [0]:
SELECT Emergency_Services
, COUNT(*) AS n
FROM hospital_analysis
GROUP BY Emergency_Services
ORDER BY n DESC
;



In [0]:
SELECT Size_Indicator
, COUNT(*) AS n
FROM hospital_analysis
GROUP BY Size_Indicator
ORDER BY Size_Indicator
;



In [0]:
SELECT Value_Quadrant
, COUNT(*) AS n
FROM hospital_analysis
GROUP BY Value_Quadrant
ORDER BY n DESC
;



In [0]:
-- Continuous summary stats
SELECT COUNT(*) AS n
, ROUND(AVG(Outcomes),3) AS avg_outcomes
, ROUND(median(Outcomes),3) AS med_outcomes
, ROUND(stddev_samp(Outcomes),3) AS std_outcomes
, ROUND(min(Outcomes),3) AS min_outcomes
, ROUND(max(Outcomes),3) AS max_outcomes
, ROUND(median(Outcomes),3) AS med_outcomes
FROM hospital_analysis
;



In [0]:
SELECT COUNT(*) AS n
, ROUND(AVG(Per_Episode_Cost),0) AS avg_cost
, ROUND(stddev_samp(Per_Episode_Cost),0) AS std_cost
, ROUND(min(Per_Episode_Cost),0) AS min_cost
, ROUND(max(Per_Episode_Cost),0) AS max_cost
, ROUND(median(Per_Episode_Cost),0) AS med_cost
FROM hospital_analysis
;



In [0]:
-- Net tilt: this is a single value representing the difference between the number of hospitals in the 'high outcomes, low cost' category and the 'low outcomes, high cost' category

CREATE OR REPLACE TEMP VIEW net_tilt_census AS
WITH q AS (
    SELECT Census_Division
    , Value_Quadrant
    , COUNT(*) AS n
    FROM hospital_value
    GROUP BY Census_Division
    , Value_Quadrant
    )
SELECT Census_Division
, ROUND(SUM(CASE WHEN Value_Quadrant = 'high outcome , low cost'     THEN n 
          ELSE 0 
          END) / SUM(n)
    - SUM(CASE WHEN Value_Quadrant = 'low outcome , high cost' THEN n 
          ELSE 0 
          END) / SUM(n),3)
    AS net_tilt
FROM q
GROUP BY Census_Division
ORDER BY net_tilt DESC
;

SELECT * FROM net_tilt_census; 



In [0]:
CREATE OR REPLACE TEMP VIEW net_tilt_ownership AS
WITH q AS (
    SELECT Hospital_Ownership
    , Value_Quadrant
    , COUNT(*) AS n
    FROM hospital_value
    GROUP BY Hospital_Ownership
    , Value_Quadrant
    )
SELECT Hospital_Ownership
, ROUND(SUM(CASE WHEN Value_Quadrant = 'high outcome , low cost'     THEN n 
          ELSE 0 
          END) / SUM(n) 
    - SUM(CASE WHEN Value_Quadrant = 'low outcome , high cost' THEN n 
          ELSE 0 
          END) / SUM(n),3) 
    AS net_tilt
FROM q
GROUP BY hospital_ownership
ORDER BY net_tilt DESC
;

SELECT * FROM net_tilt_ownership;

## Layer 8: Advanced analyses

In [0]:
-- Locating the top 3 facilities per census division according to outcomes, with overall national rank listed

SELECT RANK() OVER (ORDER BY Outcomes DESC) Overall_Rank
, Census_Division
, Facility_Name
, City_Town
, State
, ROUND(Outcomes,3)
FROM hospital_value
QUALIFY DENSE_RANK() OVER (PARTITION BY Census_Division ORDER BY Outcomes DESC) <= 3
ORDER BY Census_Division, Outcomes DESC
;

In [0]:
-- Number of facilities in each value quadrant by census division

SELECT Census_Division
, Value_Quadrant
, COUNT(*) AS n
FROM hospital_value
GROUP BY ROLLUP (Census_Division, Value_Quadrant)
;